In [1]:
import os
import pandas as pd
import seaborn as sns
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go
import matplotlib.pyplot as plt

from hbn.constants import Defaults
from hbn.visualization import visualize

import warnings
warnings.filterwarnings("ignore")

%load_ext autoreload
%autoreload 2

%matplotlib inline

pio.renderers.default = 'iframe'

In [20]:
# load model output

df_classify = pd.read_csv(os.path.join(Defaults.MODEL_DIR, 'classifier-all-phenotypic-models-performance.csv'))
df_regress = pd.read_csv(os.path.join(Defaults.MODEL_DIR, 'regression-all-phenotypic-models-performance.csv'))

df_classify['data'] = df_classify['data'].map({'model-data': 'data', 'model-null': 'null'})

for idx,col in enumerate(['Assessment', 'Domain', 'Measure']):
    df_classify[col] = df_classify['features'].str.split('-').str.get(idx)

In [21]:
df_feature = pd.read_csv(os.path.join(Defaults.MODEL_DIR, 'classifier-feature_importance.csv'))

df_feature = df_feature[df_feature['Measure']!='all']

In [4]:
# set plotting style
visualize.plotting_style()

In [22]:
def interactive_plot(df, x='features', y='roc_auc_score'):
    
    fig = go.Figure()

    fig.add_trace(go.Violin(x=df[x][df['data']=='data'],
                            y=df[y][df['data']=='data'],
                            legendgroup='Null', scalegroup='Null', name='Null',
                            side='negative',
                            line_color='blue')
                 )
    fig.add_trace(go.Violin(x=df[x][df['data']=='null'],
                            y=df[y][df['data']=='null'],
                            legendgroup='Data', scalegroup='Data', name='Data',
                            side='positive',
                            line_color='orange')
                 )
    fig.update_traces(meanline_visible=True, box_visible=False)
    fig.update_layout(violingap=0, violinmode='overlay')
    fig.update_xaxes(showticklabels=False)
    fig.update_yaxes(title_text=y)
    fig.show()
    
def stable_plot(df, x='features', y='roc_auc_score', hue='data'):
    ax = sns.violinplot(
        x=x, 
        y=y, 
        hue=hue, 
        data=df, 
        split=True,  
        scale='width',
        inner='quartile'
        )
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.1),
              ncol=2, fancybox=True, shadow=True)
    plt.xticks(rotation=90)
    plt.show()

In [23]:
# Child Measures - Language Tasks

df1 = df_classify[(df_classify['target']=='DX_01_Cat_binarize') & 
                  (df_classify['Assessment']=='Child_Measures') & 
                  (df_classify['Domain']=='Language_Tasks')]


interactive_plot(df=df1)

In [24]:
# Parent Measures - Demographics

df1 = df_classify[(df_classify['target']=='DX_01_Cat_binarize') & 
                  (df_classify['Assessment']=='Parent_Measures') & 
                  (df_classify['Domain']=='Demographic_Questionnaire_Measures')]


interactive_plot(df=df1)

In [25]:
# Parent Measures - Interview of Emotional and Psychological Function

df1 = df_classify[(df_classify['target']=='DX_01_Cat_binarize') & 
                  (df_classify['Assessment']=='Parent_Measures') & 
                  (df_classify['Domain']=='Interview_of_Emotional_and_Psychological_Function')]


interactive_plot(df=df1)

In [26]:
# Child Measures - Cognitive Testing

df1 = df_classify[(df_classify['target']=='DX_01_Cat_binarize') & 
                  (df_classify['Assessment']=='Child_Measures') & 
                  (df_classify['Domain']=='Cognitive_Testing')]


interactive_plot(df=df1)

In [27]:
# Child Measures - Questionnaire_Measures_of_Emotional_and_Cognitive_Status

df1 = df_classify[(df_classify['target']=='DX_01_Cat_binarize') & 
                  (df_classify['Assessment']=='Child_Measures') & 
                  (df_classify['Domain']=='Questionnaire_Measures_of_Emotional_and_Cognitive_Status')]


interactive_plot(df=df1)

In [28]:
# Child Measures : all other domains

df1 = df_classify[(df_classify['target']=='DX_01_Cat_binarize') & 
                  (df_classify['Assessment']=='Child_Measures')]

cols = ['Physical_Fitness_and_Status', 'Neurologic_Function', 'Physiologic_Function', 'Vision', 'Motor_Skills']

df1 = df1[df1['Domain'].isin(cols)]

interactive_plot(df=df1)

In [13]:
## feature importance per model

for name, group in df_feature.groupby(['features']):
    
    group = group.reset_index()
    fig = px.line(group, x='feature_names_sum', y='feature_sum', markers=True, title=name)
    fig.update_xaxes(showticklabels=False)
    fig.update_xaxes(type='category')
    
    fig.show()